In [2]:
import numpy as np
import pandas as pd

from scipy.stats import norm
import matplotlib.pyplot as plt

Our objective is to price an Asian option with a fixed strike price.

In [9]:
def GBM(S_0, r, T, sigma, no_paths= 100):

  no_steps = 365
  dt = T/no_steps

  np.random.seed(10)

  Z = np.random.normal(0, 1, [no_paths, no_steps])
  X = np.zeros([no_paths, no_steps + 1])
  time = np.zeros([no_steps + 1])

  X[:, 0] = np.log(S_0)

  for i in range(no_steps):
    if no_paths > 1:
      Z[:, i] = (Z[:, i] - np.mean(Z[:, i])) / np.std(Z[:, i])
    X[:, i + 1] = X[:, i] + (r - 0.5 * np.square(sigma)) * dt + sigma * np.sqrt(dt) * Z[:, i]
    time[i + 1] = time[i] + dt

  S = np.exp(X)

  paths = {'time': time, 'S': S}

  return paths

In [19]:
def Asian_Option_pricing(S_0, r, T, sigma, type, no_paths= 1000):

  path = GBM(S_0, r, T, sigma, no_paths)
  S_mean = path['S'].mean(axis= 1)

  if type == "call":
    payoff = np.maximum(S_mean - K, 0)
  elif type == "put":
    payoff = np.maximum(K - S_mean, 0)
  else:
    raise ValueError("Option type should be either 'call' or 'put' ")

  price = np.exp(-r * T) * np.mean(payoff)
  stdErr = np.exp(-r * T) * np.std(payoff, ddof= 1)/np.sqrt(no_paths)

  return price, stdErr

In [20]:
r = 0.045
K = 110
sigma = 0.35
T = 1
S_0 = 100

In [21]:
Asian_Option_pricing(S_0, r, T, sigma, 'call')

(np.float64(5.525983839850948), np.float64(0.3726461577146893))

In [22]:
no_paths = 10
while no_paths <= 10**6:
  price, stderr = Asian_Option_pricing(S_0, r, T, sigma, 'call', no_paths)
  print(f"The price of Asian Option is:  {price}")
  print(f"The standard error is:  {stderr}")
  no_paths *= 10

The price of Asian Option is:  4.2760382054531245
The standard error is:  2.1492973959919293
The price of Asian Option is:  5.224976115929801
The standard error is:  1.188888113855111
The price of Asian Option is:  5.525983839850948
The standard error is:  0.3726461577146893
The price of Asian Option is:  5.049944121531233
The standard error is:  0.10843335394664805
The price of Asian Option is:  5.074506807819675
The standard error is:  0.034880163382257585
The price of Asian Option is:  5.08528594986573
The standard error is:  0.011073054285815552


In [8]:
print(f"{'Paths':>10}  {'MC Price':>10}  {'Std Err':>10}  {'95% CI Low':>12}  {'95% CI High':>12}")

     Paths    MC Price     Std Err    95% CI Low   95% CI High
